# MNIST Surface Experiment

## Imports

In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"

import znrnd
import numpy as np

from dask_jobqueue import SLURMCluster
from dask.distributed import Client

2023-01-20 08:47:20.598895: W tensorflow/compiler/xla/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libnvinfer.so.7'; dlerror: libnvinfer.so.7: cannot open shared object file: No such file or directory
2023-01-20 08:47:20.598984: W tensorflow/compiler/xla/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libnvinfer_plugin.so.7'; dlerror: libnvinfer_plugin.so.7: cannot open shared object file: No such file or directory
2023-01-20 08:47:20.598992: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Cannot dlopen some TensorRT libraries. If you would like to use Nvidia GPU with TensorRT, please make sure the missing libraries mentioned above are installed properly.
2023-01-20 08:47:22.920945: W external/org_tensorflow/tensorflow/tsl/platform/default/dso_loader.cc:66] Could not load dynamic library 'libcuda.so.1'; dlerror: libcuda.so.1: cannot open shared object file: No such file or directory
2023-01

## Expose the experiment classes

In [2]:
%run full-entropy-eigenvalue-surface.ipynb

## Fixed parameters

These are parameters that are fixed for MNIST and will always be constant for the experiments. These include:

* `data_generator`: Where to get data from.
* `input_shape`: Fixed image size.
* `output_size`: 10 classes are predicted.
* `flatten`: The inputs must be flattened.
* `accuracy`: We will use an accuracy function.
* `root_path`: Where to save the data.
* `test_ds_size`: 500 is standard

In [3]:
data_generator = znrnd.data.MNISTGenerator
input_shape = (1, 28, 28, 1)
output_size = 10
flatten = True
accuracy = True
root_path = "MNIST-Dense-NTK"
test_ds_size = 200

## Prepare the inputs

In [4]:
def compute_closest_divisble_number(samples: np.ndarray, divisor: int):
    """
    Compute the number closes to N that divides evenly by divisor.
    """
    clipped_data = np.zeros_like(samples)
    
    def _compute_value(n, m):
        """
        vectorized oepration.
        """
        q = int(n / m)

        # 1st possible closest number
        n1 = m * q

        # 2nd possible closest number
        if((n * m) > 0) :
            n2 = (m * (q + 1))
        else :
            n2 = (m * (q - 1))

        # if true, then n1 is the required closest number
        if (abs(n - n1) < abs(n - n2)) :
            return n1

        # else n2 is the required closest number
        return n2
    
    for i, item in enumerate(samples):
        clipped_data[i] = _compute_value(item, 5)
#         clipped_data[i] = gcd(item, divisor)

    return clipped_data

In [5]:
def build_input_variants(
    n_samples: int, 
    data_generator=data_generator,
    input_shape=input_shape, 
    output_size=output_size, 
    flatten=flatten, 
    accuracy=accuracy,
    root_path=root_path,
    test_ds_size=test_ds_size,
    offset: int = 0
):
    """
    Build the input arrays for a fixed sample size.
    
    Parameters
    ----------
    n_samples : int
            Number of points that will be in the surface.
    """
    # Get values that require computations
    ds_sizes = compute_closest_divisble_number(
        np.random.randint(low=200, high=1000, size=(n_samples,)), 5
    )
    network_depths = [2] * n_samples #np.random.randint(low=1, high=5, size=(n_samples,))
    layer_widths = [128] * n_samples #np.random.randint(low=12, high=300, size=(n_samples,))
    indices = np.linspace(0 + offset, n_samples + offset, n_samples, dtype=int)

    # Scale the fixed values
    data_generator = [data_generator] * n_samples
    input_shape = [input_shape] * n_samples
    output_size = [output_size] * n_samples
    flatten = [flatten] * n_samples
    accuracy = [accuracy] * n_samples
    root_path = [root_path] * n_samples
    test_ds_size = [test_ds_size] * n_samples
    
    return (
        ds_sizes, 
        network_depths, 
        layer_widths, 
        data_generator,
        input_shape, 
        output_size, 
        flatten, 
        accuracy,
        root_path,
        test_ds_size,
        indices
    )

## Run the experiment

In [6]:
cluster = SLURMCluster(
    cores=64, 
    memory="1TB",
    queue="cpu-long",
    processes=8,
    name="Dense-Surface",
    job_extra=["--time=24:00:00"]
)

/home/ac134186/miniconda3/envs/zincware/lib/python3.10/site-packages/dask_jobqueue/core.py:255: FutureWarning: job_extra has been renamed to job_extra_directives. You are still using it (even if only set to []; please also check config files). If you did not set job_extra_directives yet, job_extra will be respected for now, but it will be removed in a future release. If you already set job_extra_directives, job_extra is ignored and you can remove it.
  warnings.warn(warn, FutureWarning)
/home/ac134186/miniconda3/envs/zincware/lib/python3.10/site-packages/distributed/node.py:182: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 45313 instead
  warnings.warn(


In [15]:
cluster2 = SLURMCluster(
    cores=64, 
    memory="1TB",
    queue="cpu",
    processes=8,
    name="Dense-Surface",
    job_extra=["--time=24:00:00"]
)

/home/ac134186/miniconda3/envs/zincware/lib/python3.10/site-packages/dask_jobqueue/core.py:255: FutureWarning: job_extra has been renamed to job_extra_directives. You are still using it (even if only set to []; please also check config files). If you did not set job_extra_directives yet, job_extra will be respected for now, but it will be removed in a future release. If you already set job_extra_directives, job_extra is ignored and you can remove it.
  warnings.warn(warn, FutureWarning)
/home/ac134186/miniconda3/envs/zincware/lib/python3.10/site-packages/distributed/node.py:182: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 36837 instead
  warnings.warn(


In [16]:
cluster2.adapt(minimum_jobs=0, maximum_jobs=20)

In [7]:
cluster.adapt(minimum_jobs=0, maximum_jobs=20)

/home/ac134186/miniconda3/envs/zincware/lib/python3.10/site-packages/dask_jobqueue/core.py:255: FutureWarning: job_extra has been renamed to job_extra_directives. You are still using it (even if only set to []; please also check config files). If you did not set job_extra_directives yet, job_extra will be respected for now, but it will be removed in a future release. If you already set job_extra_directives, job_extra is ignored and you can remove it.
  warnings.warn(warn, FutureWarning)


In [17]:
(
    ds_sizes, 
    network_depths, 
    layer_widths, 
    data_generators,
    input_shapes, 
    output_sizes, 
    flattens, 
    accuracies,
    root_paths,
    test_ds_sizes,
    indices
) = build_input_variants(5000, offset=5000)

In [9]:
client = Client(cluster)

In [18]:
client2 = Client(cluster2)

In [10]:
client

Connection method: Cluster object,Cluster type: dask_jobqueue.SLURMCluster
Dashboard: http://129.69.187.97:45313/status,
Dashboard: http://129.69.187.97:45313/status,Workers: 0
Total threads: 0,Total memory: 0 B
Comm: tcp://129.69.187.97:46035,Workers: 0
Dashboard: http://129.69.187.97:45313/status,Total threads: 0
Started: Just now,Total memory: 0 B


In [19]:
client2

Connection method: Cluster object,Cluster type: dask_jobqueue.SLURMCluster
Dashboard: http://129.69.187.97:36837/status,
Dashboard: http://129.69.187.97:36837/status,Workers: 0
Total threads: 0,Total memory: 0 B
Comm: tcp://129.69.187.97:40139,Workers: 0
Dashboard: http://129.69.187.97:36837/status,Total threads: 0
Started: Just now,Total memory: 0 B


In [20]:
# results_array = []

results_array.append(client2.map(
    run_experiment, 
    ds_sizes, 
    network_depths,
    layer_widths, 
    data_generators, 
    input_shapes,
    output_sizes,
    flattens,
    accuracies,
    test_ds_sizes,
    root_paths,
    indices
)
)

In [23]:
cluster2.close()

/home/ac134186/miniconda3/envs/zincware/lib/python3.10/site-packages/dask_jobqueue/core.py:255: FutureWarning: job_extra has been renamed to job_extra_directives. You are still using it (even if only set to []; please also check config files). If you did not set job_extra_directives yet, job_extra will be respected for now, but it will be removed in a future release. If you already set job_extra_directives, job_extra is ignored and you can remove it.
  warnings.warn(warn, FutureWarning)


In [14]:
client.gather(results_array)

RuntimeError: instance allocation failed: new instance has no pybind11-registered base types